# 01 — Data Exploration

**Purpose:** ingest and clean the raw workbook, validate schema/data quality, and persist the canonical daily dataset. No modeling dependencies belong in this notebook.


In [1]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / "src"))


In [2]:
from pathlib import Path
import os

# Interactive data-source configuration. Set GAME_DATA_FILE in the environment
# to override this path on another machine.
SOURCE_FILE = Path(
    os.getenv(
        "GAME_DATA_FILE",
        r"C:\Users\BaneleS\OneDrive - HollywoodBets\Santh Munsamy's files - Slots Game Data\Tournament Daily Dataset Feb26-Jul26.xlsx",
    )
)
os.environ["GAME_DATA_FILE"] = str(SOURCE_FILE)

from game_performance.config import SETTINGS
from game_performance.data import load_raw_workbook, clean_game_daily, data_quality_summary
from game_performance.io import save_dataframe

SETTINGS.ensure_directories()
print("Source:", SETTINGS.source_file)
if not SETTINGS.source_file.exists():
    raise FileNotFoundError(
        f"Dataset not found: {SETTINGS.source_file}\n"
        "Update SOURCE_FILE in this notebook or set GAME_DATA_FILE to the workbook location."
    )



Source: C:\Users\BaneleS\OneDrive - HollywoodBets\Santh Munsamy's files - Slots Game Data\Tournament Daily Dataset Feb26-Jul26.xlsx


In [3]:
# Load the first worksheet exactly as in the source notebook,
# while also showing available sheets for validation.

xls = pd.ExcelFile(SETTINGS.source_file)

print("Available sheets:", xls.sheet_names)

raw_df = pd.read_excel(
    SETTINGS.source_file,
    sheet_name=xls.sheet_names[0],
)

print(f"Loaded sheet: {xls.sheet_names[0]}")
print(f"Raw shape: {raw_df.shape}")

display(raw_df.head())


Available sheets: ['Sheet1']
Loaded sheet: Sheet1
Raw shape: (880062, 9)


,YearMonth,Date,SystemName,TournamentName,Clients,Betcount,Stakes,GGR,GGR_Perc
0,2026-02-Feb,2026-02-01,AmusnetSlots,AmusnetS-100 Bulky Fruits,121,12010,61820.0,-1715.10,-0.0277
1,2026-02-Feb,2026-02-01,AmusnetSlots,AmusnetS-100 Cats,10373,674056,1514739.0,35209.29,0.0232
2,2026-02-Feb,2026-02-01,AmusnetSlots,AmusnetS-100 Power Hot,9516,749111,1398001.0,50178.39,0.0358
3,2026-02-Feb,2026-02-01,AmusnetSlots,AmusnetS-2 Dragons,211,14456,36738.2,-3390.90,-0.0922
4,2026-02-Feb,2026-02-01,AmusnetSlots,AmusnetS-20 Bulky Fruits,2449,173855,400810.6,-7301.83,-0.0182


In [4]:
df = clean_game_daily(
    raw_df,
    invalid_identifier_policy="drop",
)

summary = data_quality_summary(df)

excluded = df.attrs.get(
    "excluded_invalid_identifier_rows",
    0,
)

print(f"Raw rows: {len(raw_df):,}")
print(f"Clean rows: {len(df):,}")
print(f"Excluded invalid identifier rows: {excluded:,}")

print("\nDate range:")
display(summary["date_range"])

Raw rows: 880,062
Clean rows: 880,049
Excluded invalid identifier rows: 13

Date range:


,min_date,max_date,games
0,2026-02-01,2026-07-29,22622


In [5]:
display(summary["missing"].head(20))
display(summary["numeric_summary"])


,missing_count
GGR_Perc_Calc,22494
Date,0
SystemName,0
TournamentName,0
YearMonth,0
Clients,0
Betcount,0
GGR,0
Stakes,0
GGR_Perc,0


,count,mean,std,min,25%,50%,75%,max
Clients,880049.0,1006.348828,6997.086787,1.0,18.0,72.0,304.0,431798.0
Betcount,880049.0,153532.984502,2278138.121555,1.0,404.0,3929.0,24206.0,177026358.0
Stakes,880049.0,373394.762235,5374442.337839,0.0,1806.0,12569.8,68337.0,538530538.79
GGR,880049.0,14879.232141,280006.042519,-60163540.7725,0.0,633.8,4343.0,33358737.63
GGR_Perc,880049.0,-53.738154,1914.560639,-497200.0,0.0,0.0761,0.23,1.0
Days_Since_First_Observed,880049.0,76.598566,51.568233,1.0,31.0,72.0,118.0,179.0
GGR_Perc_Calc,857555.0,-0.016607,11.914838,-9750.26728,0.008618,0.080454,0.238201,1.0


In [6]:
output_path = save_dataframe(df, SETTINGS.processed_dir / "clean_game_daily.pkl")
print(f"Saved {len(df):,} cleaned rows to {output_path}")


Saved 880,049 cleaned rows to C:\Users\BaneleS\Downloads\Game Performance Model\artifacts\processed\clean_game_daily.pkl
